In [1]:
import os
import importlib
import pickle
import json
import copy
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from together import Together

import time
from datetime import timedelta
from tqdm import tqdm

import math_parser 
importlib.reload(math_parser)
from math_parser import *

In [2]:
def save_object(x, file_path):
    with open(file_path, 'wb') as file:
        pickle.dump(x, file)

def load_object(file_path):
    with open(file_path, 'rb') as file:
        x = pickle.load(file)
    return x

def save_json(x, file_path):
    with open(file_path, 'w') as json_file:
        json.dump(x, json_file, indent=4)

def load_json(file_path):
    with open(file_path, 'r') as json_file:
        x = json.load(json_file)
    return x

In [3]:
api_key = os.getenv('TogetherAI_API_Key')
client = Together(api_key=api_key)
model = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

In [4]:
# TODO!!!: when generating ver questions, create them using the whole step-wise solution instead of step by step; also can test if passing (t-1) steps, prompting the model to predict what the next atomic step would be, and then comparing with original (t) step
# TODO!!!: maybe experiment with "role": "system"; first add instructions; then do personalities like math professor (very knowledgable); instruction follower for few-shot prompts (concise);confident personality vs. very thorough, detailed, skeptical personality 
# TODO!!: could also use deepseek?
# TODO!!: test effects of number of verification questions (maybe make it a parameter either aboslute i.e. 3 ver questions each max)
# TODO!: question sandwhich i.e. ask verification question, give context, ask ver question again
# TODO!: also generation of verification questions is pretty broad, check if diff prompts i.e. using markdown to make it more organized can help? do we need to specify Q and A or Response and Context? or just diff prompts in general 

In [ ]:
# TODO: note that to adapt to math domain where each step is reliant on the other, the verifiction part of the paper had to be slightly modified; also general structure is ordered sequentially instead of in parallel
# TODO: limitation is that final prompt is going to be huge
# TODO: Generating step by step

In [6]:
def create_user_msg(s):
    return [{"role": "user", "content": s}]

def append_user_msg(s, messages):
    messages.append({"role": "user", "content": s})

def chat_and_update_history(model, messages):
    response = client.chat.completions.create(model=model, messages=messages)
    response_context = {"role": "assistant", "content": response.choices[0].message.content}
    messages.append(response_context)

# MATH Dataset

In [7]:
math_data = pd.read_parquet('./data/MATH.parquet')
# removing '?' level
math_data = math_data[math_data['level'] != 'Level ?']
_, math_samp = train_test_split(math_data, test_size=0.0042, stratify=math_data[['level', 'type']], random_state=42)
math_ex = math_samp.sample(3, replace=False, random_state=42)
math_samp = math_samp.drop(index=math_ex.index)
math_ex = math_ex.reset_index(drop=True)
math_samp = math_samp.reset_index(drop=True)

In [111]:
math_samp['level'].value_counts()

level
Level 5    15
Level 4    13
Level 3    10
Level 2     7
Level 1     5
Name: count, dtype: int64

In [120]:
math_samp.loc[7, 'problem']

'The ones digit of the product of four consecutive positive integers is 4. If this product is greater than 1000, what is the sum of the four smallest such integers?'

In [8]:
math_samp['type'].value_counts()

type
Algebra                   11
Intermediate Algebra       9
Prealgebra                 8
Number Theory              6
Counting & Probability     6
Geometry                   5
Precalculus                5
Name: count, dtype: int64

## Baseline

In [56]:
start_time = time.time()
result = {
    'is_correct': [], 
    'response': []
}
for i in tqdm(range(math_samp.shape[0])):
    messages = create_user_msg(f"Solve the problem P and box the final, simplified answer with the LaTeX command \\boxed{{...}}.\nP: {math_samp.loc[i, 'problem']}")
    chat_and_update_history(model, messages)
    result['is_correct'].append(is_correct(math_samp.loc[i, 'solution'], messages[1]['content']))
    result['response'].append(messages[1]['content'])

end_time = time.time()
print(f"Total time: {str(timedelta(seconds=round(end_time - start_time)))}")
result = pd.DataFrame(result)

100%|██████████| 50/50 [04:08<00:00,  4.98s/it]

Total time: 0:04:09


In [58]:
# save_object(result, './result_base.pkl')
result = load_object('./base_result.pkl')

In [59]:
result[~result['is_correct']]

,is_correct,response
4,False,## Step 1: Understand the given operation\nThe...
9,False,## Step 1: To find the minimum value of the gi...
23,False,## Step 1: Understand the given product notati...
24,False,"## Step 1: Simplify the expression\nFirst, let..."
27,False,"To find the number of handshakes, we can use t..."
35,False,## Step 1: Understand the problem and the sequ...


## CoVe

In [9]:
few_shot_messages = load_json('./few_shot_examples.json')

In [26]:
# global settings
random.seed(42)
n_ver_questions = 2
n_break_step = 10
cove_result = {
    'is_correct': [], 
    'response': []
}

In [27]:
start_time = time.time()
for i in tqdm(range(math_samp.shape[0])):
    # setting up variables for the problem
    problem = f"P: {math_samp.loc[i, 'problem']}"
    step_list = []
    steps = '\n\n'.join(step_list)
    step_num = 0
    is_partial_answer = True
    while is_partial_answer:
        # setting up conditionals for the loop
        is_partial_answer = not "\\boxed" in steps
        if (not is_partial_answer) or step_num >= n_break_step:
            break
        if step_num == 0:
            answer_str = "" 
        elif is_partial_answer:
            answer_str = " and solution A"

        # generate new step
        gen_step_messages = copy.deepcopy(few_shot_messages)
        gen_step_messages = gen_step_messages['gen_step']
        gen_step_instruct = f"Given a problem P{answer_str}, generate only the next atomic step (Step {step_num+1}). Box the final, simplified answer with the LaTeX command \\boxed{{...}}. Do not give the final answer unless this is truly the last step. Do not output more than one step."
        gen_step_pa = f"{problem}"
        if step_num != 0:
            gen_step_pa += f"\nA: {steps}"
        gen_step_input = '\n'.join([gen_step_instruct, gen_step_pa])
        append_user_msg(gen_step_input, gen_step_messages)
        chat_and_update_history(model, gen_step_messages)

        # update steps
        step_list.append(gen_step_messages[-1]['content'])
        steps = '\n\n'.join(step_list)

        # generating verification step
        gen_ver_messages = copy.deepcopy(few_shot_messages)
        gen_ver_messages = gen_ver_messages['gen_ver']
        gen_ver_instruct = f"Given a problem P{answer_str}, write verification question(s) to validate Step {step_num+1}. Only output the questions as a semicolon separated list and ensure that each question is self-contained and independent of each other."
        gen_ver_qa = f"{problem}\nA: {steps}"
        gen_ver_input = '\n'.join([gen_ver_instruct, gen_ver_qa])
        append_user_msg(gen_ver_input, gen_ver_messages)
        chat_and_update_history(model, gen_ver_messages)

        # sampling verification questions
        ver_questions = gen_ver_messages[-1]['content'].split('; ')
        if n_ver_questions < len(ver_questions):
            ver_questions = random.sample(ver_questions, n_ver_questions)

        # cutting out step to verify
        ver_steps = '\n\n'.join(step_list[:-1])

        # verify and revise step
        for j in range(len(ver_questions)):
            # answering verifying questions independently
            ver_messages = copy.deepcopy(few_shot_messages)
            ver_messages = ver_messages['ver']
            ver_instruct = f"Concisely answer the question given the context of problem P{answer_str}."
            ver_qc = f"Question: {ver_questions[j]}\nContext: {problem}"
            if step_num != 0:
                ver_qc += f"\nA: {ver_steps}"
            ver_input = '\n'.join([ver_instruct, ver_qc])
            append_user_msg(ver_input, ver_messages)
            chat_and_update_history(model, ver_messages)

            # revising step using verification question and answer
            rev_messages = copy.deepcopy(few_shot_messages)
            rev_messages = rev_messages['rev']
            rev_instruct = f"Output if Step {step_num+1} is \"CONSISTENT\" or \"INCONSISTENT\" with the given context. Only output \"INCONSISTENT\" if Step {step_num+1} needs to be revised. If \"INCONSISTENT\", then re-write Step {step_num+1} with the same format."
            rev_co = f"Context: Q: {ver_questions[j]}\nA: {ver_messages[-1]['content']}\nOriginal: {problem}\nA: {steps}"
            rev_input = '\n'.join([rev_instruct, rev_co])
            append_user_msg(rev_input, rev_messages)
            chat_and_update_history(model, rev_messages)

            # checking for inconsistency and revising
            if "INCONSISTENT" in rev_messages[-1]['content']:
                new_step = '\n\n'.join(rev_messages[-1]['content'].split('\n\n')[1:])
                step_list[-1] = new_step 
                steps = '\n\n'.join(step_list)
        
        # moving on to generate next step
        step_num += 1

    # if stuck in loop, then mark the question as wrong
    if step_num >= n_break_step:
        steps = ""
    
    cove_result['response'].append(steps)
    cove_result['is_correct'].append(is_correct(math_samp.loc[i, 'solution'], steps))

end_time = time.time()
print(f"Total time: {str(timedelta(seconds=round(end_time - start_time)))}")
cove_result = pd.DataFrame(cove_result)

100%|██████████| 50/50 [39:51<00:00, 47.83s/it]  

Total time: 0:39:52


In [28]:
save_object(cove_result, './cove_result.pkl')

In [41]:
i = 2

In [ ]:
# setting up variables for the problem
problem = f"P: {math_samp.loc[i, 'problem']}"
step_list = []
steps = '\n\n'.join(step_list)
step_num = 0
is_partial_answer = True
while is_partial_answer:
    # setting up conditionals for the loop
    is_partial_answer = not "\\boxed" in steps
    if (not is_partial_answer) or step_num >= n_break_step:
        break
    if step_num == 0:
        answer_str = "" 
    elif is_partial_answer:
        answer_str = " and solution A"

    # generate new step
    gen_step_messages = copy.deepcopy(few_shot_messages)
    gen_step_messages = gen_step_messages['gen_step']
    gen_step_instruct = f"Given a problem P{answer_str}, generate only the next atomic step (Step {step_num+1}). Box the final, simplified answer with the LaTeX command \\boxed{{...}}. Do not give the final answer unless this is truly the last step. Do not output more than one step."
    gen_step_pa = f"{problem}"
    if step_num != 0:
        gen_step_pa += f"\nA: {steps}"
    gen_step_input = '\n'.join([gen_step_instruct, gen_step_pa])
    append_user_msg(gen_step_input, gen_step_messages)
    chat_and_update_history(model, gen_step_messages)

    # update steps
    step_list.append(gen_step_messages[-1]['content'])
    steps = '\n\n'.join(step_list)

    # generating verification step
    gen_ver_messages = copy.deepcopy(few_shot_messages)
    gen_ver_messages = gen_ver_messages['gen_ver']
    gen_ver_instruct = f"Given a problem P{answer_str}, write verification question(s) to validate Step {step_num+1}. Only output the questions as a semicolon separated list and ensure that each question is self-contained and independent of each other."
    gen_ver_qa = f"{problem}\nA: {steps}"
    gen_ver_input = '\n'.join([gen_ver_instruct, gen_ver_qa])
    append_user_msg(gen_ver_input, gen_ver_messages)
    chat_and_update_history(model, gen_ver_messages)

    # sampling verification questions
    ver_questions = gen_ver_messages[-1]['content'].split('; ')
    if n_ver_questions < len(ver_questions):
        ver_questions = random.sample(ver_questions, n_ver_questions)

    # cutting out step to verify
    ver_steps = '\n\n'.join(step_list[:-1])

    # verify and revise step
    for j in range(len(ver_questions)):
        # answering verifying questions independently
        ver_messages = copy.deepcopy(few_shot_messages)
        ver_messages = ver_messages['ver']
        ver_instruct = f"Concisely answer the question given the context of problem P{answer_str}."
        ver_qc = f"Question: {ver_questions[j]}\nContext: {problem}"
        if step_num != 0:
            ver_qc += f"\nA: {ver_steps}"
        ver_input = '\n'.join([ver_instruct, ver_qc])
        append_user_msg(ver_input, ver_messages)
        chat_and_update_history(model, ver_messages)

        # revising step using verification question and answer
        rev_messages = copy.deepcopy(few_shot_messages)
        rev_messages = rev_messages['rev']
        rev_instruct = f"Output if Step {step_num+1} is \"CONSISTENT\" or \"INCONSISTENT\" with the given context. Only output \"INCONSISTENT\" if Step {step_num+1} needs to be revised. If \"INCONSISTENT\", then re-write Step {step_num+1} with the same format."
        rev_co = f"Context: Q: {ver_questions[j]}\nA: {ver_messages[-1]['content']}\nOriginal: {problem}\nA: {steps}"
        rev_input = '\n'.join([rev_instruct, rev_co])
        append_user_msg(rev_input, rev_messages)
        chat_and_update_history(model, rev_messages)

        # checking for inconsistency and revising
        if "INCONSISTENT" in rev_messages[-1]['content']:
            new_step = '\n\n'.join(rev_messages[-1]['content'].split('\n\n')[1:])
            step_list[-1] = new_step 
            steps = '\n\n'.join(step_list)
    
    # moving on to generate next step
    step_num += 1

    print(step_list[-1])

# if stuck in loop, then mark the question as wrong
if step_num >= n_break_step:
    steps = "stuck in loop"


## Step 1: Understand the constraints of the polynomial
The polynomial is of the form $x^9 + a_8 x^8 + a_7 x^7 + \dots + a_2 x^2 + a_1 x + a_0$, where each coefficient $a_i$ can only be 0 or 1, and we are looking for polynomials with exactly two distinct integer roots.
## Step 2: Recognize that the possible integer roots are limited
Since the polynomial has integer coefficients and we are looking for integer roots, the possible roots must divide the constant term $a_0$. Given $a_0$ can only be 0 or 1, the possible integer roots are limited to $\pm 1$ when $a_0 = 1$, because 0 does not divide any number and the only divisors of 1 are $\pm 1$. If $a_0 = 0$, then 0 is also a root, but we're focusing on non-zero integer roots for this step.
## Step 3: Determine the conditions for having exactly two distinct integer roots
For a polynomial to have exactly two distinct integer roots, given the constraints, if $a_0 = 0$, then the polynomial has $x = 0$ as a root. This means it cannot have exac

AttributeError: 'Series' object has no attribute 'append'

In [45]:
math_samp.loc[2, 'solution']

'If all the $a_i$ are equal to 0, then the polynomial becomes $x^9 = 0,$ which has only one integer root, namely $x = 0.$  Thus, we can assume that there is some coefficient $a_i$ that is non-zero.  Let $k$ be the smallest integer such that $a_k \\neq 0$; then we can take out a factor of $x^k,$ to get\n\\[x^k (x^{9 - k} + a_8 x^{8 - k} + a_7 x^{7 - k} + \\dots + a_{k + 1} x + a_k) = 0.\\]By the Integer Root Theorem, any integer root of $x^{9 - k} + a_8 x^{8 - k} + \\dots + a_{k + 1} x + a_k = 0$ must divide $a_k = 1,$ so the only possible integer roots are 1 and $-1.$ However, if we plug in $x = 1,$ we see that $x^{9 - k} = 1,$ and all the other terms are nonnegative, so $x = 1$ cannot be a root.\n\nTherefore, for the original polynomial to have two different integer roots, they must be 0 and $-1.$  For 0 to be a root, it suffices to take $a_0 = 0,$ and the polynomial is\n\\[x^9 + a_8 x^8 + a_7 x^7 + a_6 x^6 + a_5 x^5 + a_4 x^4 + a_3 x^3 + a_2 x^2 + a_1 x = 0.\\]We also want $x = -1$ t

In [ ]:
cove_result.loc[, 'response'].split('\n\n')

['## Step 1: Recall the formula for the area of a triangle\nThe area $A$ of a triangle is given by the formula $A = \\frac{1}{2}bh$, where $b$ is the length of the base and $h$ is the altitude (height) of the triangle. Given that the area $A$ is 600 square feet and the length of the corresponding base $b$ is 30 feet, we can use this formula to solve for the altitude $h$.',
 '## Step 2: Substitute the given values into the formula and solve for the altitude\nSubstituting $A = 600$ and $b = 30$ into the formula $A = \\frac{1}{2}bh$, we get $600 = \\frac{1}{2} \\cdot 30 \\cdot h$. Simplifying this equation gives $600 = 15h$. To solve for $h$, we divide both sides of the equation by 15, yielding $h = \\frac{600}{15} = 40$. Therefore, the altitude of the triangle is $h = 40$ feet, which can be expressed as $\\boxed{40}$.']

In [52]:
cove_result['num_steps'] = cove_result['response'].apply(lambda x: len(x.split('\n\n')))

In [124]:
sum(cove_result['is_correct'] & result['is_correct'])

39

In [125]:
sum(~cove_result['is_correct'] & ~result['is_correct'])

4

In [127]:
sum((cove_result['is_correct'] ^ result['is_correct']) & (result['is_correct']))

5

In [131]:
sum(cove_result['is_correct']) / len(cove_result['is_correct'])

0.82

In [129]:
sum(result['is_correct'])

44

In [128]:
sum(result['is_correct']) / len(result['is_correct'])

0.88

In [ ]:
# TODO: structured output
# TODO: mark steps that have been revised

In [109]:
problem_num = 35
print(parse_and_normalize(math_samp.loc[problem_num, 'solution']))
print(math_samp.loc[problem_num, 'level'])

2^{2008}-2
Level 5


In [110]:
cove_result.loc[problem_num, 'response'].split('\n\n')

['## Step 1: The problem asks whether the hyperbola $\\mathcal{C}$ given by the equation $y^2 - x^2 = 1$ is indeed a hyperbola, to which the answer is yes, confirming that the equation represents a hyperbola.',
 '## Step 2: The sequence of points $(P_n)$ is constructed by finding the intersection of the line $\\ell_n$ with the hyperbola $\\mathcal{C}$ and then taking the orthogonal projection of this intersection point onto the $x$-axis, which is a consistent description of the process given in the problem.',
 '## Step 3: The behavior of the sequence $(P_n)$ and the construction of $P_{n+1}$ from $P_n$ indeed depend on whether the starting point $P_0$ is on the positive $x$-axis or the negative $x$-axis, as the intersection of the line $\\ell_n$ with the hyperbola $\\mathcal{C}$ and the subsequent orthogonal projection onto the $x$-axis will differ based on the initial position of $P_0$.',
 '## Step 4: The sequence $(P_n)$ has a period that divides 2008 if and only if $P_0 = P_{2008}$.

In [67]:
cove_result[~cove_result['is_correct']]

,is_correct,response,num_steps
2,False,## Step 1: Understand the constraints of the p...,7
4,False,,1
7,False,## Step 1: Understand the problem and identify...,5
10,False,## Step 1: Identify the range of integers\nThe...,5
13,False,## Step 1: Identify the given equation and the...,2
22,False,## Step 1: To find the largest prime factor of...,7
23,False,## Step 1: Confirm the range of the product\nT...,5
24,False,## Step 1: Simplify the given expression using...,5
35,False,## Step 1: The problem asks whether the hyperb...,6


In [31]:
sum(cove_result['is_correct']) / len(cove_result['is_correct'])

0.82